# Objective 2 – Economic & Policy Implications of SAF Deployment

This notebook builds directly on the **Objective 1 official output** to evaluate the
economic implications of Sustainable Aviation Fuel (SAF) deployment in the EU27
aviation sector.

We:
1. Load the Objective 1 metrics dataset (`Year`, `Scenario`, `Total_Fuel`, `SAF_Share`, `CO2_Emissions`, `Avoided_CO2`).
2. Apply our economic assumptions:
   - Jet fuel and SAF prices (€/tonne)
   - EU ETS carbon price trajectory (€/tCO₂)
   - SAF lifecycle CO₂ reduction (~75%)
3. Compute, by year and scenario:
   - Fuel costs (jet, SAF, total)
   - Carbon costs under EU ETS
   - ETS savings from avoided emissions
   - Fossil-only counterfactual fuel cost
   - Marginal abatement cost (MAC) in €/tCO₂

The outputs from this notebook will feed directly into:
- The **Executive Summary (Objective 2)** and
- The **Final Presentation (Objective 3)**.



In [1]:
import pathlib

import numpy as np
import pandas as pd

# Display options for clarity in the notebook
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


In [6]:
# Path to Objective 1 output

df_obj1 = pd.read_csv("../data/metrics/obj1_official_output__eu27__annual__2026-2050.csv")


print("Objective 1 metrics – head():")
display(df_obj1.head())

print("\nDataFrame info():")
df_obj1.info()


Objective 1 metrics – head():


,Year,Scenario,Total_Fuel,SAF_Share,CO2_Emissions,Avoided_CO2
0,2026,0,38.50,1.40,120.38,1.28
1,2026,1,38.50,3.80,118.19,3.47
2,2027,0,38.50,1.80,120.02,1.64
3,2027,1,38.50,5.60,116.55,5.11
4,2028,0,38.50,2.20,119.65,2.01



DataFrame info():
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Year           50 non-null     int64  
 1   Scenario       50 non-null     int64  
 2   Total_Fuel     50 non-null     float64
 3   SAF_Share      50 non-null     float64
 4   CO2_Emissions  50 non-null     float64
 5   Avoided_CO2    50 non-null     float64
dtypes: float64(4), int64(2)
memory usage: 2.5 KB


## Economic Assumptions

We use the following **core parameters**, based on public sources:

- **Jet fuel price (Jet A-1)**  
  - 800 €/tonne in 2025, increasing modestly over time.

- **SAF price**  
  - 2,200 €/tonne in 2025 (≈ 2.75× conventional jet fuel), with a gradual cost reduction towards 2050 as the market scales.

- **EU ETS carbon price**  
  - 70 €/tCO₂ in 2025, rising to 140 €/tCO₂ by 2035 and 180 €/tCO₂ by 2050.

- **SAF lifecycle CO₂ reduction**  
  - 75% vs fossil jet fuel (SAF_CO2_reduction = 0.75), consistent with the 70–80% range suggested in the Datathon brief.

These values will be used to compute:
- Fuel expenditures,
- ETS-related carbon costs and savings,
- Marginal abatement cost (MAC) in €/tCO₂.


In [7]:
# ---- Assumption anchors ----

# Years present in the Objective 1 dataset
years = df_obj1["Year"].unique()
years.sort()

year_min, year_max = years.min(), years.max()
year_min, year_max


(2026, 2050)

In [8]:
# 1) JET FUEL PRICE TRAJECTORY (€/tonne)
# Anchor: 2025 -> 800 €/t, 2050 -> 950 €/t (gentle increase)

jet_price_years = np.array([2025, 2050])
jet_price_vals = np.array([800.0, 950.0])  # €/t

def jet_price_eur_per_tonne(year: np.ndarray) -> np.ndarray:
    return np.interp(year, jet_price_years, jet_price_vals)


# 2) SAF PRICE TRAJECTORY (€/tonne)
# Anchor: 2025 -> 2,200 €/t, 2050 -> 1,800 €/t (learning curve, costs decline)

saf_price_years = np.array([2025, 2050])
saf_price_vals = np.array([2200.0, 1800.0])  # €/t

def saf_price_eur_per_tonne(year: np.ndarray) -> np.ndarray:
    return np.interp(year, saf_price_years, saf_price_vals)


# 3) CARBON PRICE TRAJECTORY (€/tCO2)
# Anchor: 2025 -> 70 €/t, 2035 -> 140 €/t, 2050 -> 180 €/t

carbon_price_years = np.array([2025, 2035, 2050])
carbon_price_vals = np.array([70.0, 140.0, 180.0])  # €/tCO2

def carbon_price_eur_per_tco2(year: np.ndarray) -> np.ndarray:
    return np.interp(year, carbon_price_years, carbon_price_vals)


# 4) SAF LIFECYCLE CO2 REDUCTION (dimensionless, 0–1)
SAF_CO2_REDUCTION = 0.75  # 75%
SAF_CO2_REDUCTION


0.75

In [9]:
df = df_obj1.copy()

# Convert units
df["Total_Fuel_tonnes"] = df["Total_Fuel"] * 1_000_000.0
df["SAF_fraction"] = df["SAF_Share"] / 100.0

df["SAF_tonnes"] = df["Total_Fuel_tonnes"] * df["SAF_fraction"]
df["Jet_tonnes"] = df["Total_Fuel_tonnes"] - df["SAF_tonnes"]

# Attach prices based on year
df["Jet_Price_EUR_per_tonne"] = jet_price_eur_per_tonne(df["Year"].values)
df["SAF_Price_EUR_per_tonne"] = saf_price_eur_per_tonne(df["Year"].values)
df["Carbon_Price_EUR_per_tCO2"] = carbon_price_eur_per_tco2(df["Year"].values)

# Fuel cost components
df["Jet_Fuel_Cost_EUR"] = df["Jet_tonnes"] * df["Jet_Price_EUR_per_tonne"]
df["SAF_Fuel_Cost_EUR"] = df["SAF_tonnes"] * df["SAF_Price_EUR_per_tonne"]
df["Total_Fuel_Cost_EUR"] = df["Jet_Fuel_Cost_EUR"] + df["SAF_Fuel_Cost_EUR"]

# CO2 and ETS-related costs
df["CO2_tonnes"] = df["CO2_Emissions"] * 1_000_000.0
df["Avoided_CO2_tonnes"] = df["Avoided_CO2"] * 1_000_000.0

df["Carbon_Cost_EUR"] = df["CO2_tonnes"] * df["Carbon_Price_EUR_per_tCO2"]
df["ETS_Savings_EUR"] = df["Avoided_CO2_tonnes"] * df["Carbon_Price_EUR_per_tCO2"]

# Fossil-only counterfactual: all fuel is jet fuel at jet price
df["Fossil_Only_Fuel_Cost_EUR"] = df["Total_Fuel_tonnes"] * df["Jet_Price_EUR_per_tonne"]

# Extra fuel cost due to SAF
df["Extra_SAF_Fuel_Cost_EUR"] = df["Total_Fuel_Cost_EUR"] - df["Fossil_Only_Fuel_Cost_EUR"]

# Net abatement cost after ETS savings
df["Net_Abatement_Cost_EUR"] = df["Extra_SAF_Fuel_Cost_EUR"] - df["ETS_Savings_EUR"]

# Marginal Abatement Cost (€/tCO2)
# Avoid division by zero
df["MAC_EUR_per_tCO2"] = df["Net_Abatement_Cost_EUR"] / df["Avoided_CO2_tonnes"].replace(0, np.nan)

print("Enriched Objective 1 dataset with economic metrics:")
display(df.head())


Enriched Objective 1 dataset with economic metrics:


,Year,Scenario,Total_Fuel,SAF_Share,CO2_Emissions,Avoided_CO2,Total_Fuel_tonnes,SAF_fraction,SAF_tonnes,Jet_tonnes,...,SAF_Fuel_Cost_EUR,Total_Fuel_Cost_EUR,CO2_tonnes,Avoided_CO2_tonnes,Carbon_Cost_EUR,ETS_Savings_EUR,Fossil_Only_Fuel_Cost_EUR,Extra_SAF_Fuel_Cost_EUR,Net_Abatement_Cost_EUR,MAC_EUR_per_tCO2
0,2026,0,38.50,1.40,120.38,1.28,"38,500,000.00",0.01,"539,000.00","37,961,000.00",...,"1,177,176,000.00","31,773,742,000.00","120,383,000.00","1,277,000.00","9,269,491,000.00","98,329,000.00","31,031,000,000.00","742,742,000.00","644,413,000.00",504.63
1,2026,1,38.50,3.80,118.19,3.47,"38,500,000.00",0.04,"1,463,000.00","37,037,000.00",...,"3,195,192,000.00","33,047,014,000.00","118,193,000.00","3,467,000.00","9,100,861,000.00","266,959,000.00","31,031,000,000.00","2,016,014,000.00","1,749,055,000.00",504.49
2,2027,0,38.50,1.80,120.02,1.64,"38,500,000.00",0.02,"693,000.00","37,807,000.00",...,"1,502,424,000.00","32,201,708,000.00","120,018,000.00","1,642,000.00","10,081,512,000.00","137,928,000.00","31,262,000,000.00","939,708,000.00","801,780,000.00",488.29
3,2027,1,38.50,5.60,116.55,5.11,"38,500,000.00",0.06,"2,156,000.00","36,344,000.00",...,"4,674,208,000.00","34,185,536,000.00","116,550,000.00","5,110,000.00","9,790,200,000.00","429,240,000.00","31,262,000,000.00","2,923,536,000.00","2,494,296,000.00",488.12
4,2028,0,38.50,2.20,119.65,2.01,"38,500,000.00",0.02,"847,000.00","37,653,000.00",...,"1,822,744,000.00","32,622,898,000.00","119,653,000.00","2,007,000.00","10,888,423,000.00","182,637,000.00","31,493,000,000.00","1,129,898,000.00","947,261,000.00",471.98


In [10]:
print("Scenarios present:", df["Scenario"].unique())

print("\nTotal fuel cost by scenario (sum over all years):")
display(
    df.groupby("Scenario")["Total_Fuel_Cost_EUR"].sum().to_frame("Total_Fuel_Cost_EUR")
)

print("\nTotal carbon cost by scenario (sum over all years):")
display(
    df.groupby("Scenario")["Carbon_Cost_EUR"].sum().to_frame("Total_Carbon_Cost_EUR")
)

print("\nTotal avoided CO2 and net abatement cost (sum over all years):")
summary = df.groupby("Scenario")[["Avoided_CO2_tonnes", "Net_Abatement_Cost_EUR"]].sum()
summary["Average_MAC_EUR_per_tCO2"] = (
    summary["Net_Abatement_Cost_EUR"] / summary["Avoided_CO2_tonnes"].replace(0, np.nan)
)
display(summary)

Scenarios present: [0 1]

Total fuel cost by scenario (sum over all years):


,Total_Fuel_Cost_EUR
Scenario,
0,"998,874,415,000.00"
1,"1,207,695,335,000.00"



Total carbon cost by scenario (sum over all years):


,Total_Carbon_Cost_EUR
Scenario,
0,"368,592,968,666.67"
1,"294,944,855,333.33"



Total avoided CO2 and net abatement cost (sum over all years):


,Avoided_CO2_tonnes,Net_Abatement_Cost_EUR,Average_MAC_EUR_per_tCO2
Scenario,,,
0,"362,243,000.00","95,973,943,666.67",264.94
1,"838,543,000.00","231,146,710,333.33",275.65


In [11]:
columns_for_obj2 = [
    "Year",
    "Scenario",
    "Total_Fuel",
    "SAF_Share",
    "CO2_Emissions",
    "Avoided_CO2",
    "Total_Fuel_tonnes",
    "SAF_tonnes",
    "Jet_tonnes",
    "Jet_Price_EUR_per_tonne",
    "SAF_Price_EUR_per_tonne",
    "Carbon_Price_EUR_per_tCO2",
    "Jet_Fuel_Cost_EUR",
    "SAF_Fuel_Cost_EUR",
    "Total_Fuel_Cost_EUR",
    "Fossil_Only_Fuel_Cost_EUR",
    "Extra_SAF_Fuel_Cost_EUR",
    "Carbon_Cost_EUR",
    "ETS_Savings_EUR",
    "Net_Abatement_Cost_EUR",
    "MAC_EUR_per_tCO2",
]

df_obj2 = df[columns_for_obj2].copy()

print("Objective 2 economic metrics – head():")
display(df_obj2.head())


Objective 2 economic metrics – head():


,Year,Scenario,Total_Fuel,SAF_Share,CO2_Emissions,Avoided_CO2,Total_Fuel_tonnes,SAF_tonnes,Jet_tonnes,Jet_Price_EUR_per_tonne,...,Carbon_Price_EUR_per_tCO2,Jet_Fuel_Cost_EUR,SAF_Fuel_Cost_EUR,Total_Fuel_Cost_EUR,Fossil_Only_Fuel_Cost_EUR,Extra_SAF_Fuel_Cost_EUR,Carbon_Cost_EUR,ETS_Savings_EUR,Net_Abatement_Cost_EUR,MAC_EUR_per_tCO2
0,2026,0,38.50,1.40,120.38,1.28,"38,500,000.00","539,000.00","37,961,000.00",806.00,...,77.00,"30,596,566,000.00","1,177,176,000.00","31,773,742,000.00","31,031,000,000.00","742,742,000.00","9,269,491,000.00","98,329,000.00","644,413,000.00",504.63
1,2026,1,38.50,3.80,118.19,3.47,"38,500,000.00","1,463,000.00","37,037,000.00",806.00,...,77.00,"29,851,822,000.00","3,195,192,000.00","33,047,014,000.00","31,031,000,000.00","2,016,014,000.00","9,100,861,000.00","266,959,000.00","1,749,055,000.00",504.49
2,2027,0,38.50,1.80,120.02,1.64,"38,500,000.00","693,000.00","37,807,000.00",812.00,...,84.00,"30,699,284,000.00","1,502,424,000.00","32,201,708,000.00","31,262,000,000.00","939,708,000.00","10,081,512,000.00","137,928,000.00","801,780,000.00",488.29
3,2027,1,38.50,5.60,116.55,5.11,"38,500,000.00","2,156,000.00","36,344,000.00",812.00,...,84.00,"29,511,328,000.00","4,674,208,000.00","34,185,536,000.00","31,262,000,000.00","2,923,536,000.00","9,790,200,000.00","429,240,000.00","2,494,296,000.00",488.12
4,2028,0,38.50,2.20,119.65,2.01,"38,500,000.00","847,000.00","37,653,000.00",818.00,...,91.00,"30,800,154,000.00","1,822,744,000.00","32,622,898,000.00","31,493,000,000.00","1,129,898,000.00","10,888,423,000.00","182,637,000.00","947,261,000.00",471.98


In [12]:
OBJ2_OUTPUT_PATH = REPO_ROOT / "data" / "metrics" / "obj2_economic_metrics__eu27__annual__2026-2050.csv"

OBJ2_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_obj2.to_csv(OBJ2_OUTPUT_PATH, index=False)

print(f"Saved Objective 2 metrics to: {OBJ2_OUTPUT_PATH}")

Saved Objective 2 metrics to: C:\Users\neome\iata-datathon-2\notebooks\data\metrics\obj2_economic_metrics__eu27__annual__2026-2050.csv


In [13]:
# Aggregate to scenario-level 2026–2050
agg = (
    df_obj2
    .groupby("Scenario")
    .agg(
        Total_Fuel_Mt=("Total_Fuel", "sum"),
        Total_CO2_Mt=("CO2_Emissions", "sum"),
        Total_Avoided_CO2_Mt=("Avoided_CO2", "sum"),
        Total_Fuel_Cost_EUR=("Total_Fuel_Cost_EUR", "sum"),
        Total_Carbon_Cost_EUR=("Carbon_Cost_EUR", "sum"),
        Total_ETS_Savings_EUR=("ETS_Savings_EUR", "sum"),
        Total_Net_Abatement_Cost_EUR=("Net_Abatement_Cost_EUR", "sum"),
    )
)

agg["Average_MAC_EUR_per_tCO2"] = (
    agg["Total_Net_Abatement_Cost_EUR"]
    / (agg["Total_Avoided_CO2_Mt"] * 1_000_000.0).replace(0, np.nan)
)

# For readability: express big € amounts in billions
for col in ["Total_Fuel_Cost_EUR", "Total_Carbon_Cost_EUR", "Total_ETS_Savings_EUR", "Total_Net_Abatement_Cost_EUR"]:
    agg[col.replace("_EUR", "_EUR_billion")] = agg[col] / 1e9

display(agg)

,Total_Fuel_Mt,Total_CO2_Mt,Total_Avoided_CO2_Mt,Total_Fuel_Cost_EUR,Total_Carbon_Cost_EUR,Total_ETS_Savings_EUR,Total_Net_Abatement_Cost_EUR,Average_MAC_EUR_per_tCO2,Total_Fuel_Cost_EUR_billion,Total_Carbon_Cost_EUR_billion,Total_ETS_Savings_EUR_billion,Total_Net_Abatement_Cost_EUR_billion
Scenario,,,,,,,,,,,,
0,962.50,"2,679.26",362.24,"998,874,415,000.00","368,592,968,666.67","57,825,471,333.33","95,973,943,666.67",264.94,998.87,368.59,57.83,95.97
1,962.50,"2,202.96",838.54,"1,207,695,335,000.00","294,944,855,333.33","131,473,624,666.67","231,146,710,333.33",275.65,"1,207.70",294.94,131.47,231.15
